In [4]:
import pandas as pd
import numpy as np

# 1. Chargement du jeu d'entraînement
path = "../data/UNSW_NB15_training-set.parquet"
df = pd.read_parquet(path)

initial_rows = df.shape[0]
initial_cols = df.shape[1]

In [10]:
# 2. Suppression des colonnes inutiles ou sources de fuite de données (Data Leakage)
# - 'id' : identifiant séquentiel sans valeur prédictive (si présent)
# - 'attack_cat' : cible multi-classes qui révélerait directement la réponse binaire
cols_to_drop = ['id', 'attack_cat']
existing_drop_cols = [col for col in cols_to_drop if col in df.columns]

df_clean = df.drop(columns=existing_drop_cols)
print(f"Colonnes supprimées : {existing_drop_cols}")

Colonnes supprimées : ['attack_cat']


In [11]:
# 3. Traitement des valeurs infinies (inf / -inf) et manquantes (NaN)
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)

nan_count = df_clean.isnull().sum().sum()
if nan_count > 0:
    df_clean = df_clean.dropna()
    print(f"Lignes contenant des NaN/Inf supprimées : {nan_count}")
else:
    print("Aucune valeur NaN ou infinie détectée.")

Aucune valeur NaN ou infinie détectée.


In [12]:
# 4. Suppression des doublons exacts
duplicates_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Doublons exacts supprimés : {duplicates_count:,}")

Doublons exacts supprimés : 82,984


In [13]:
# 5. Bilan du nettoyage
print("\n=== BILAN DU NETTOYAGE ===")
print(f"Dimensions initiales : {initial_rows:,} lignes x {initial_cols} colonnes")
print(f"Dimensions finales   : {df_clean.shape[0]:,} lignes x {df_clean.shape[1]} colonnes")
print(f"Rétention des données : {(df_clean.shape[0] / initial_rows * 100):.2f}%")


=== BILAN DU NETTOYAGE ===
Dimensions initiales : 175,341 lignes x 36 colonnes
Dimensions finales   : 92,357 lignes x 35 colonnes
Rétention des données : 52.67%


In [14]:
df_clean.to_parquet("../data/UNSW_NB15_train_clean.parquet", index=False)
print("Dataset nettoyé sauvegardé avec succès.")

Dataset nettoyé sauvegardé avec succès.


In [15]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
# 1. Séparer les variables explicatives (X) et la cible (y)
X = df_clean.drop(columns=['label'])
y = df_clean['label']

# 2. Identifier les colonnes catégorielles et numériques
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Colonnes catégorielles à encoder : {cat_cols}")
for col in cat_cols:
    print(f" - {col} : {X[col].nunique()} catégories uniques")

# 3. Configurer le One-Hot Encoder
# handle_unknown='ignore' évite tout crash si le jeu de test contient un protocole inédit
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 4. Appliquer l'encodage tout en conservant les colonnes numériques intactes
ct_encoder = ColumnTransformer(
    transformers=[
        ('cat', encoder, cat_cols)
    ],
    remainder='passthrough'  # Laisse passer les colonnes numériques sans les modifier pour l'instant
)

X_encoded_array = ct_encoder.fit_transform(X)

# Reconstruire un DataFrame lisible pour vérifier le résultat
encoded_feature_names = ct_encoder.get_feature_names_out()
X_encoded = pd.DataFrame(X_encoded_array, columns=encoded_feature_names)

print(f"\nDimensions avant encodage : {X.shape}")
print(f"Dimensions après encodage : {X_encoded.shape}")
display(X_encoded.head(3))

Colonnes catégorielles à encoder : ['proto', 'service', 'state']
 - proto : 133 catégories uniques
 - service : 13 catégories uniques
 - state : 9 catégories uniques

Dimensions avant encodage : (92357, 34)
Dimensions après encodage : (92357, 186)


,cat__proto_3pc,cat__proto_a/n,cat__proto_aes-sp3-d,cat__proto_any,cat__proto_argus,cat__proto_aris,cat__proto_arp,cat__proto_ax.25,cat__proto_bbn-rcc,cat__proto_bna,...,remainder__smean,remainder__dmean,remainder__trans_depth,remainder__response_body_len,remainder__ct_src_dport_ltm,remainder__ct_dst_sport_ltm,remainder__is_ftp_login,remainder__ct_ftp_cmd,remainder__ct_flw_http_mthd,remainder__is_sm_ips_ports
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,43.0,43.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,52.0,1106.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,46.0,824.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Séparation Train (80%) / Validation (20%) stratifiée avant toute normalisation
X_train_enc, X_val_enc, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Cibler uniquement les 31 colonnes numériques d'origine (préfixées par 'remainder__')
# On laisse intactes les colonnes binaires (0/1) issues du One-Hot Encoding
num_encoded_cols = [col for col in X_encoded.columns if col.startswith('remainder__')]

scaler = StandardScaler()

X_train_scaled = X_train_enc.copy()
X_val_scaled = X_val_enc.copy()

# 3. Ajuster (fit) UNIQUEMENT sur X_train, puis transformer X_train et X_val
X_train_scaled[num_encoded_cols] = scaler.fit_transform(X_train_enc[num_encoded_cols])
X_val_scaled[num_encoded_cols] = scaler.transform(X_val_enc[num_encoded_cols])

# 4. Vérification statistique de la normalisation
print("=== VÉRIFICATION DE LA NORMALISATION (X_train_scaled) ===")
print(f"Dimensions X_train_scaled : {X_train_scaled.shape}")
print(f"Dimensions X_val_scaled   : {X_val_scaled.shape}")
display(X_train_scaled[num_encoded_cols[:5]].describe().loc[['mean', 'std', 'min', 'max']].round(3))

=== VÉRIFICATION DE LA NORMALISATION (X_train_scaled) ===
Dimensions X_train_scaled : (73885, 186)
Dimensions X_val_scaled   : (18472, 186)


,remainder__dur,remainder__spkts,remainder__dpkts,remainder__sbytes,remainder__dbytes
mean,0.000,0.000,0.000,0.000,0.000
std,1.000,1.000,1.000,1.000,1.000
min,-0.266,-0.178,-0.242,-0.066,-0.145
max,11.413,50.639,75.345,53.130,76.990
